# Beautiful Shop — BeautifulSoup와 Selenium
**미션:** 배너의 “전 상품 절반 세일!”은 사실일까?

| 단계 | 도구 | 누가 |
|---|---|---|
| ① 첫 화면 상품 세기 | BeautifulSoup | 모두 실행 |
| ② 「상품 더보기」 누르고 다시 세기 | Selenium | 강사 시연 |
| ③ 로그인해야 보이는 회원가 | Selenium | 강사 시연 |

실행: 오른쪽 위 **커널 선택 → `.venv`** → 위에서부터 ▶ (자세한 방법은 같은 폴더의 `사용방법.md`)

In [1]:
# ▶ 준비 확인 — 연습 파일이 이 노트북과 같은 폴더에 있는지
from pathlib import Path
SHOP  = Path('shop.html').resolve()
LOGIN = Path('login.html').resolve()
for f in [SHOP, LOGIN]:
    print('✅' if f.exists() else '❌ 없음 →', f.name)

✅ shop.html
✅ login.html


## ① BeautifulSoup — 첫 화면에서 SALE 상품 세기

In [2]:
from bs4 import BeautifulSoup

soup  = BeautifulSoup(SHOP.read_text(encoding='utf-8'), 'html.parser')
items = soup.select('li.item')              # 상품 카드 전부
sale  = soup.select('li.item .sale')        # SALE 배지 전부

for item in items:
    name  = item.select_one('.name').text
    price = item.select_one('.price').contents[0]
    print(name, price)

print(f'SALE {len(sale)}개 / 전체 {len(items)}개 = {len(sale)/len(items):.0%}')

머그컵 12,000원
목도리 25,000원
노트 3,500원
향초 18,000원
SALE 2개 / 전체 4개 = 50%


결과는 50% — 광고가 맞는 것 같아요. 그런데 화면 맨 아래 **「상품 더보기」** 버튼 뒤 상품은 shop.html 파일에 처음부터 들어 있지 않아서 BeautifulSoup은 못 봐요.

## ② Selenium — 버튼을 눌러 진짜 비율 확인 (강사 시연)
실행하면 크롬 창이 열리고 버튼이 저절로 눌려요.

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()                      # 크롬 열기 (드라이버는 자동으로 받아요)
driver.get(SHOP.as_uri())                        # 같은 폴더의 shop.html 열기

items = driver.find_elements(By.CSS_SELECTOR, 'li.item')
print('처음 상품 수:', len(items))

driver.find_element(By.ID, 'more').click()       # 「상품 더보기」 클릭
WebDriverWait(driver, 10).until(                 # 버튼이 사라질 때까지 = 상품이 다 뜰 때까지
    EC.invisibility_of_element_located((By.ID, 'more')))

items = driver.find_elements(By.CSS_SELECTOR, 'li.item')
sale  = driver.find_elements(By.CSS_SELECTOR, 'li.item .sale')
print('클릭 후 상품 수:', len(items))
print(f'SALE {len(sale)}개 / 전체 {len(items)}개 = {len(sale)/len(items):.0%}')

처음 상품 수: 4
클릭 후 상품 수: 8
SALE 2개 / 전체 8개 = 25%


### 클릭 후 HTML을 BeautifulSoup으로 넘겨 표로 만들기

In [ ]:
import pandas as pd
soup = BeautifulSoup(driver.page_source, 'html.parser')   # 지금 화면의 HTML
rows = [{'상품명': i.select_one('.name').text,
         '가격': i.select_one('.price').contents[0],
         'SALE': i.select_one('.sale') is not None} for i in soup.select('li.item')]
driver.save_screenshot('after.png')
driver.quit()                                    # 다 쓰면 꼭 닫기
pd.DataFrame(rows)

,상품명,가격,SALE
0,머그컵,"12,000원",False
1,목도리,"25,000원",True
2,노트,"3,500원",False
3,향초,"18,000원",True
4,양말,"5,000원",False
5,미니 화분,"9,900원",False
6,에코백,"15,000원",False
7,텀블러,"22,000원",False


**팩트체크 결과:** 광고는 50%, 실제는 25% — 첫 화면만 수집하면 틀린 기사가 돼요.

## ③ 로그인해야 보이는 회원가 (강사 시연)
연습 계정: 아이디 `reporter` · 비밀번호 `beautiful123` — 실행하면 위쪽에 비밀번호 입력칸이 떠요.

In [ ]:
from getpass import getpass
driver = webdriver.Chrome()
driver.get(LOGIN.as_uri())
driver.find_element(By.ID, 'username').send_keys('reporter')
driver.find_element(By.ID, 'password').send_keys(getpass('비밀번호: '))   # 코드에 비밀번호를 쓰지 않기
############################################ 직접 입력해서도 실행해 보기!!!
driver.find_element(By.ID, 'login').click()

WebDriverWait(driver, 10).until(                 # 회원가가 뜰 때까지
    EC.presence_of_element_located((By.CLASS_NAME, 'member-price')))
for item in driver.find_elements(By.CSS_SELECTOR, 'li.item'):
    print(item.find_element(By.CLASS_NAME, 'name').text,
          item.find_element(By.CLASS_NAME, 'member-price').text)
driver.quit()

머그컵 회원가 9,000원
목도리 회원가 20,000원
텀블러 회원가 15,000원
